# Amazon ML Challenge 2026: Business Entity Resolution
## 01. Exploratory Data Analysis & Prototyping

This notebook covers the baseline data exploration, entity clustering distributions, inverted-index blocking prototyping, and feature engineering for cross-source entity matching across Source 1 (Anchor), Source 2, and Source 3.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure code/src is accessible
sys.path.append(os.path.abspath("../business_entity_resolution/code/src"))
from preprocess import clean_text, canonicalize_entity
from blocking import InvertedIndexBlocker
from evaluate import compute_macro_f05

### 1. Data Ingestion & Schema Inspection

In [ ]:
train_s1 = pd.read_csv("../dataset/train/train_source1.tsv", sep="\t")
train_s2 = pd.read_csv("../dataset/train/train_source2.tsv", sep="\t")
train_s3 = pd.read_csv("../dataset/train/train_source3.tsv", sep="\t")
train_gt = pd.read_csv("../dataset/train/train_ground_truth.tsv", sep="\t")

print(f"Source 1 shape: {train_s1.shape}")
print(f"Source 2 shape: {train_s2.shape}")
print(f"Source 3 shape: {train_s3.shape}")
print(f"Ground Truth shape: {train_gt.shape}")
display(train_s1.head())

### 2. Singleton vs Linked Entity Distribution Analysis
A key component of Macro F_0.5 is singleton handling: empty matches must remain empty to achieve full credit.

In [ ]:
# Inspect ground truth singleton ratio
train_gt['matches'] = train_gt['matches'].fillna('')
num_singletons = (train_gt['matches'] == '').sum()
total_anchors = len(train_gt)
print(f"Total S1 Entities: {total_anchors}")
print(f"True Singletons: {num_singletons} ({num_singletons / total_anchors * 100:.2f}%)")
print(f"Linked Clusters: {total_anchors - num_singletons} ({(total_anchors - num_singletons) / total_anchors * 100:.2f}%)")

### 3. Preprocessing & Legal Entity Normalization

In [ ]:
sample_entity = train_s1.iloc[0].to_dict()
canonical_sample = canonicalize_entity(sample_entity)
print("Raw Entity:", sample_entity)
print("Canonicalized Entity:", canonical_sample)

### 4. Inverted Index Blocking Prototyping
Evaluating candidate recall and pair reduction ratio with strict candidate bounds (<= 20 candidates per anchor).

In [ ]:
blocker = InvertedIndexBlocker(max_candidates=20)
aux_df = pd.concat([train_s2, train_s3], ignore_index=True)
blocker.fit(aux_df)
candidates = blocker.query(train_s1)

print(f"Generated candidate lists for {len(candidates)} Source 1 entities")
sample_s1_id = train_s1['id'].iloc[0]
print(f"Candidates for {sample_s1_id}: {candidates.get(sample_s1_id, [])}")

### 5. Pairwise Feature Engineering Summary
Extracting 27 discriminative signals across lexical, token, phonetic, geographic, and domain dimensions.

In [ ]:
from feature_extraction import extract_pair_features

if len(candidates.get(sample_s1_id, [])) > 0:
    cand_id = candidates[sample_s1_id][0]
    cand_row = aux_df[aux_df['id'] == cand_id].iloc[0].to_dict()
    feat_vec = extract_pair_features(sample_entity, cand_row)
    print(f"Feature vector dimension: {len(feat_vec)}")
    print("Feature values:", feat_vec)